In [1]:
# https://supervised.mljar.com/api/

In [2]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
%cd /content/gdrive/MyDrive/DrugRepurposing/Teza

/content/gdrive/MyDrive/DrugRepurposing/Teza


In [4]:
! pwd

/content/gdrive/MyDrive/DrugRepurposing/Teza


In [5]:
! pip install mljar-supervised

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached jedi-0.19.1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.1/540.1 kB 32.8 MB/s eta 0:00:00
Using cached jedi-0.19.1-py2.py3-none-any.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.8/362.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 7.2 MB/s eta 0:00:00
  Cr

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from supervised.automl import AutoML

from drug_repurposing import DrugRepurposing
from config import DRUG_REPURPOSING_DS_FILE_NAME, LABEL_COLUMN, DRUG_NAME_COLUMN

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.



In [7]:
# Build drug_repurposing.csv data set

DrugRepurposing()._build_data_set()

In [8]:
drug_repurposing_df = pd.read_csv(DRUG_REPURPOSING_DS_FILE_NAME)

In [9]:
drug_repurposing_df.head(10)

,Unnamed: 0,DRUG_NAME,COVID_1,COVID_2,COVID_3,COVID_4,COVID_5,COVID_6,COVID_7,COVID_8,...,COVID_25,COVID_26,COVID_27,COVID_28,COVID_29,COVID_30,COVID_31,COVID_32,COVID_33,IN_CLINICAL_TRIALS
0,0,CAD106,0.070808,0.284321,-0.075679,0.083675,-0.106010,0.137888,0.228245,-0.067327,...,-0.067287,-0.227109,0.131697,-0.176451,0.261538,0.204680,0.215153,0.117485,0.167385,False
1,1,Muromonab,0.717732,0.919284,0.640964,0.191355,0.717411,0.543829,0.547635,0.340884,...,1.231544,0.709571,0.724296,0.727924,0.463092,0.273228,0.242386,0.718242,0.802182,False
2,2,Ciprofloxacin,0.881671,1.090093,0.450489,1.032128,-0.260742,0.662196,-0.305851,-0.113766,...,1.350120,-1.300965,1.373952,1.693683,0.815727,0.836641,0.676660,1.024395,1.031143,False
3,3,Oxamic Acid,-1.352185,-1.439691,-1.585192,-1.563383,-2.009221,-1.735532,-1.486347,-1.123673,...,-2.702104,-1.042485,-2.260824,-2.995054,-2.379602,-2.265301,-1.711725,-1.808089,-1.637826,False
4,4,Pipotiazine,1.177862,1.114245,1.349202,1.658111,1.345648,1.007031,1.510499,1.186465,...,1.117293,0.597189,1.416500,1.179810,1.162405,1.399262,1.437812,1.007266,1.459241,False
5,5,Safinamide,1.036896,0.799887,1.191221,1.133880,1.505314,0.939798,1.164851,1.387881,...,1.024019,1.241902,0.944491,0.985705,0.542496,0.726667,1.115881,0.658341,0.765951,False
6,6,Mecasermin rinfabate,0.712927,0.716997,1.292576,0.240687,0.196810,0.761747,0.502281,0.420569,...,0.465166,0.420250,0.590466,1.199218,2.267291,1.314269,0.783550,0.834342,1.095557,False
7,7,5-Phosphoarabinonic Acid,0.088405,-0.492134,-0.490471,0.286066,-0.108409,-0.955600,-0.067125,0.097537,...,-0.774965,0.091844,-0.806125,-1.838049,-1.670360,-1.242157,0.115328,-0.507602,-1.041667,False
8,8,Pentaglyme,-2.967762,-2.439571,-2.499357,-2.668758,-2.926418,-3.115139,-2.894928,-2.354524,...,-3.094076,-2.365585,-2.760327,-3.060042,-3.581605,-4.344322,-3.444958,-2.684975,-2.573757,False
9,9,"Sodium phosphate, monobasic",0.447862,0.491959,0.642253,0.389479,0.272657,1.293401,0.038316,-0.075270,...,-0.725914,-0.712831,0.231298,0.298381,-0.393045,-1.628765,-0.070251,0.333789,0.371120,False


In [10]:
drug_repurposing_df.shape

(5228, 36)

In [14]:
print(X_train.columns)

Index(['COVID_1', 'COVID_2', 'COVID_3', 'COVID_4', 'COVID_5', 'COVID_6',
       'COVID_7', 'COVID_8', 'COVID_9', 'COVID_10', 'COVID_11', 'COVID_12',
       'COVID_13', 'COVID_14', 'COVID_15', 'COVID_16', 'COVID_17', 'COVID_18',
       'COVID_19', 'COVID_20', 'COVID_21', 'COVID_22', 'COVID_23', 'COVID_24',
       'COVID_25', 'COVID_26', 'COVID_27', 'COVID_28', 'COVID_29', 'COVID_30',
       'COVID_31', 'COVID_32', 'COVID_33'],
      dtype='object')


In [27]:
X = drug_repurposing_df.drop([LABEL_COLUMN, DRUG_NAME_COLUMN, 'Unnamed: 0'], axis=1)
y = drug_repurposing_df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [28]:
len(X_train)

4182

In [37]:
# 80-20 split ratio with  k_folds = 5

automl = AutoML(total_time_limit=3600 * 3,
                mode='Perform',
                eval_metric='auc',
                validation_strategy = {
                    "validation_type": "kfold",
                    "k_folds": 5,
                    "shuffle": True,
                    "stratify": True,
                    "random_seed": 123
                    },
                explain_level = 2)

automl.fit(X_train, y_train)

AutoML directory: AutoML_16
The task is binary_classification with evaluation metric auc
AutoML will use algorithms: ['Linear', 'Random Forest', 'LightGBM', 'Xgboost', 'CatBoost', 'Neural Network']
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'not_so_random', 'golden_features', 'insert_random_feature', 'features_selection', 'hill_climbing_1', 'hill_climbing_2', 'ensemble']
* Step simple_algorithms will try to check up to 1 model
1_Linear auc 0.810762 trained in 91.47 seconds (1-sample predict time 0.1163 seconds)
* Step default_algorithms will try to check up to 5 models
2_Default_LightGBM auc 0.699364 trained in 99.12 seconds (1-sample predict time 0.0819 seconds)
3_Default_Xgboost auc 0.728117 trained in 73.1 seconds (1-sample predict time 0.0582 seconds)
4_Default_CatBoost auc 0.803064 trained in 46.11 seconds (1-sample predict time 0.0629 seconds)
5_Default_NeuralNetwork auc 0.714186 trained in 46.07 seconds (1-sample predict time 

AutoML(eval_metric='auc', explain_level=2, mode='Perform',
       total_time_limit=10800,
       validation_strategy={'k_folds': 5, 'random_seed': 123, 'shuffle': True,
                            'stratify': True, 'validation_type': 'kfold'})

In [38]:
y_pred_test = automl.predict(X_test)

In [39]:
t = [ i for i in y_pred_test if i == True]
f = [ j for j in y_pred_test if j == False]

In [40]:
print('true=', len(t))
print('false=', len(f))

true= 34
false= 1012


In [41]:
test_score = roc_auc_score(y_test, y_pred_test)

In [42]:
test_score

0.5984538152610441

In [14]:
y_pred_train = automl.predict(X_train)

In [15]:
train_score = roc_auc_score(y_train, y_pred_train)

In [17]:
train_score

0.8216097438473129